# Questions for Nick

- SAT and SBT - make summary table for nick, nick email JC
- SUP - are meds correct? - meds sent to Nick
- ECMO - where is the raw file? make clif table
- alcohol withdrawl, seizure, increased icp - icd10 code check

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import math
from pandas.api.types import (
    is_numeric_dtype, is_bool_dtype, is_categorical_dtype,
    is_datetime64_any_dtype
)
import duckdb
import subprocess
import sys
import textwrap
import json
import os
import shutil

# Global Settings

In [ ]:
# create intermediate_outputs if needed
output_path = 'intermediate_outputs'
os.makedirs(output_path, exist_ok=True)

pd.set_option('display.max_columns', None)

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["paths"]["clif"]
raw_path = cfg["paths"]["raw"]
db_path = cfg["paths"]["db"]
timezone = cfg['R_timezone']

con = duckdb.connect(database=f'{output_path}/ltvv.duckdb')
con.execute(f"SET TimeZone = '{timezone}'")

print(f"File path: {clif_path}")
print(f"Time zone: {timezone}")

# File paths

In [ ]:
hourly_resp_path = f'{db_path}/clif_hourly_resp_support.parquet'
patient_path = f'{clif_path}/clif_patient.parquet'
hosp_path = f'{clif_path}/clif_hospitalization.parquet'
vitals_path = f"{clif_path}/clif_vitals.parquet"
provider_path = f'{clif_path}/clif_provider.parquet'
position_path = f'{clif_path}/clif_position.parquet'
meds_continuous_path = f'{clif_path}/clif_medication_admin_continuous.parquet'
meds_intermittent_path = f'{clif_path}/clif_medication_admin_intermittent.parquet'
labs_path = f'{clif_path}/clif_labs.parquet'
micro_non_culture_path = f'{clif_path}/clif_microbiology_nonculture.parquet'
ecmo_path = f'{clif_path}/clif_ecmo_mcs.parquet'
diag_path = f"{clif_path}/clif_admission_diagnosis.parquet"
resp_supp_path = f"{clif_path}/clif_respiratory_support.parquet"

# Get starter vent cohort

## Merge hourly vent and hourly provider data

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE hourly_data AS
WITH hourly_vent_data AS (
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_date) as date,
        recorded_hour,
        vent_episode_id,
        vent_episode_hour_seq,
        vent_episode_duration_hours,
        device_category as vent_device_category,
        mode_category as vent_mode_category,
        fio2_set,
        spo2,
        (spo2 / fio2_set) AS sf_ratio,
        peep_set,
        tidal_volume_set,
        tracheostomy,
        hospital_id,
        location_category,
        ibw,
        sex_category
    FROM '{hourly_resp_path}'
    WHERE device_category = 'imv'
        AND vent_episode_id = '1'
        AND vent_episode_duration_hours >= '24'
        --AND tidal_volume_set IS NOT NULL
        AND location_category = 'icu'
    ORDER BY hospitalizations_joined_id, recorded_date, recorded_hour
),
provider_data AS (
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_date) AS date,
        recorded_hour,
        prov_npi_shifted,
        prov_name_shifted,
        prov_spec_shifted
    FROM '{provider_path}'
),
patient_data AS (
    SELECT
        hospitalizations_joined_id,
        ANY_VALUE(patient_id) as mdm_link_id
    FROM '{hosp_path}'
    GROUP BY hospitalizations_joined_id
)
SELECT
    v.*,
    p.prov_npi_shifted,
    p.prov_name_shifted,
    p.prov_spec_shifted,
    pat.mdm_link_id
FROM hourly_vent_data as v
LEFT JOIN patient_data as pat USING(hospitalizations_joined_id)
LEFT JOIN provider_data AS p USING(hospitalizations_joined_id, date, recorded_hour);

SELECT * FROM hourly_data
""")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE data AS
-- 1: Get fifth vent hour
WITH fifth_hour AS (
    SELECT
        hospitalizations_joined_id,
        MIN(vent_episode_hour_seq) + 4 as vent_episode_hour_seq
    FROM hourly_data
    GROUP BY hospitalizations_joined_id
),

-- 2: Join with main data to get provider info
physician_df AS (
    SELECT d.*
    FROM hourly_data d
    INNER JOIN fifth_hour USING (hospitalizations_joined_id, vent_episode_hour_seq)
),

-- 3: Get providers who started MV on 5th hour with ≥25 cases
provider_counts AS (
    SELECT
        prov_npi_shifted,
        COUNT(*) AS count
    FROM physician_df
    GROUP BY prov_npi_shifted
    HAVING count >= 25
),
        
-- 4: Keep firstTime only if provider meets criteria
firstTime AS (
    SELECT p.*
    FROM physician_df p
    INNER JOIN provider_counts USING(prov_npi_shifted)
),

-- 5: Track day 1 date per hospitalization (used to prevent 2pm fallback on day 1)
day1_dates AS (
    SELECT hospitalizations_joined_id, date AS day1_date
    FROM physician_df
),

-- 6: Get all 2pm values
data_2pm AS (
    SELECT hd.*
    FROM hourly_data hd
    INNER JOIN provider_counts pc USING (prov_npi_shifted)
    WHERE hd.recorded_hour = 14.0
),
        
-- 7: Start with firstTime; fall back to 2pm only for days after day 1.
--    Tag day-1 rows TRUE; subsequent-day rows FALSE (used in stratified models
--    and to restrict the raw Vt subsequent-day query to correct cohort dates).
initial_data AS (
    SELECT *, TRUE AS initial_vent_day FROM firstTime
    UNION ALL
    SELECT d.*, FALSE AS initial_vent_day
    FROM data_2pm d
    LEFT JOIN firstTime f
        ON f.hospitalizations_joined_id = d.hospitalizations_joined_id
        AND f.date = d.date
    LEFT JOIN day1_dates d1
        ON d1.hospitalizations_joined_id = d.hospitalizations_joined_id
    WHERE f.hospitalizations_joined_id IS NULL
      AND d.date != d1.day1_date
)

SELECT *
FROM initial_data
ORDER BY hospitalizations_joined_id, date, recorded_hour;
SELECT * FROM data
""")

In [ ]:
# Raw tidal volume ascertainment — Task 1 (R1 Minor #6 fix)
#
# Replaces the LOCF/waterfall tidal_volume_set with the first actual raw charting
# event at the correct clinical timepoint:
#
#   Day 1:       first non-null tidal_volume_set where
#                recorded_dttm >= (first IMV record + 5 hours)
#                First IMV record = intubation proxy (no explicit intubation_dttm in CLIF)
#
#   Subsequent:  first non-null tidal_volume_set where
#                local hour (America/Chicago) >= 14:00, on the cohort calendar date
#
# Both CTEs are restricted to (hospitalization_id, date) pairs present in the
# `data` DuckDB table so only enrolled vent days are included.
#
# Edge-case guard: a hospitalization whose day-1 provider had <25 cases has no
# initial_vent_day=TRUE row in `data`. Its MIN(date) in `data` is therefore a
# subsequent-day entry. Without the guard, raw_day1 would assign a row to that
# date AND raw_subseq (which reads all NOT initial_vent_day rows) would also
# produce a row for the same date — creating a UNION ALL duplicate that inflates
# data rows after the Cell 64 merge.
# Fix: raw_subseq explicitly excludes MIN(date) via AND cd.date != d1.day1_date.

raw_vt_query = f"""
WITH day1_dates AS (
    SELECT hospitalizations_joined_id, MIN(date) AS day1_date
    FROM data
    GROUP BY hospitalizations_joined_id
),
-- First IMV record per hospitalization = intubation moment
intubation_times AS (
    SELECT
        rs.hospitalizations_joined_id,
        MIN(rs.recorded_dttm) AS intubation_dttm
    FROM '{resp_supp_path}' rs
    INNER JOIN (SELECT DISTINCT hospitalizations_joined_id FROM data) c
        USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
    GROUP BY rs.hospitalizations_joined_id
),
-- Day 1: next IMV record with non-null tidal_volume_set >= intubation_dttm + 5 hours.
-- Assigned to day1_date (= MIN(date) in `data` for this hospitalization).
raw_day1 AS (
    SELECT
        rs.hospitalizations_joined_id,
        d1.day1_date AS date,
        arg_min(rs.tidal_volume_set, rs.recorded_dttm) AS tidal_volume_set,
        arg_min(rs.tidal_volume_obs,  rs.recorded_dttm) AS tidal_volume_obs
    FROM '{resp_supp_path}' rs
    INNER JOIN intubation_times it USING (hospitalizations_joined_id)
    INNER JOIN day1_dates d1       USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.tidal_volume_set IS NOT NULL
      AND rs.recorded_dttm >= it.intubation_dttm + INTERVAL 5 HOURS
    GROUP BY rs.hospitalizations_joined_id, d1.day1_date
),
-- Subsequent days: first non-null tidal_volume_set after 14:00 local time,
-- restricted to non-day-1 cohort dates from `data`.
-- The INNER JOIN + AND cd.date != d1.day1_date guarantees zero overlap with raw_day1.
raw_subseq AS (
    SELECT
        rs.hospitalizations_joined_id,
        cd.date,
        arg_min(rs.tidal_volume_set, rs.recorded_dttm) AS tidal_volume_set,
        arg_min(rs.tidal_volume_obs,  rs.recorded_dttm) AS tidal_volume_obs
    FROM '{resp_supp_path}' rs
    INNER JOIN (
        SELECT hospitalizations_joined_id, date
        FROM data
        WHERE NOT initial_vent_day
    ) cd ON rs.hospitalizations_joined_id = cd.hospitalizations_joined_id
         AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) = cd.date
    INNER JOIN day1_dates d1 ON cd.hospitalizations_joined_id = d1.hospitalizations_joined_id
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.tidal_volume_set IS NOT NULL
      AND EXTRACT(HOUR FROM rs.recorded_dttm AT TIME ZONE '{timezone}') >= 14
      AND cd.date != d1.day1_date
    GROUP BY rs.hospitalizations_joined_id, cd.date
)
SELECT * FROM raw_day1
UNION ALL
SELECT * FROM raw_subseq
"""

raw_vt_df = con.execute(raw_vt_query).df()
print(f"raw_vt_df rows:            {len(raw_vt_df)}")
print(f"tidal_volume_set non-null: {raw_vt_df['tidal_volume_set'].notna().sum()}")
print(f"tidal_volume_set null:     {raw_vt_df['tidal_volume_set'].isna().sum()}")

# Guard: duplicates here would inflate data rows in the Cell 64 merge
_dups = raw_vt_df.duplicated(subset=['hospitalizations_joined_id', 'date']).sum()
if _dups:
    raise ValueError(f"raw_vt_df has {_dups} duplicate (hospitalization_id, date) rows — investigate")
print("Duplicate check passed.")
raw_vt_df.head()

In [ ]:
con.sql(f"SELECT COUNT(Distinct prov_npi_shifted) from data")

# Patient

In [ ]:
con.sql(f"SELECT * FROM '{patient_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW patient AS
    SELECT DISTINCT ON (mdm_link_id)
        mdm_link_id,
        sex_category,
        language_category,
        race_category,
        ethnicity_category,
        death_dttm,
        DATE(death_dttm) as death_date,
        birth_date
    FROM '{patient_path}';
SELECT * FROM patient
""")

# Height

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW height AS
    SELECT
        h.patient_id as mdm_link_id,
        ANY_VALUE(v.vital_value) as height_cm
    FROM '{vitals_path}' v
    INNER JOIN '{hosp_path}' h USING (hospitalizations_joined_id)
    WHERE v.vital_category = 'height_cm'
    GROUP BY mdm_link_id;
SELECT * FROM height
""")

# Hospitalization

In [ ]:
con.sql(f"SELECT * FROM '{hosp_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW hospitalization AS
WITH base AS (
    SELECT
        hospitalizations_joined_id,
        MIN(admission_dttm) AS admission_dttm,
        MAX(discharge_dttm) AS discharge_dttm,
        ANY_VALUE(age_at_admission) AS age,
        ANY_VALUE(elix_vw) AS elix_vw
    FROM '{hosp_path}'
    GROUP BY hospitalizations_joined_id
)
SELECT
    *,
    DATE(admission_dttm) AS admission_date,
    DATE(discharge_dttm) AS discharge_date,
    YEAR(admission_dttm) AS admit_year,
    DATEDIFF('minute', admission_dttm, discharge_dttm) AS hosp_los_min,
    DATEDIFF('minute', admission_dttm, discharge_dttm) / 60.0 AS hosp_los_hr,
    DATEDIFF('minute', admission_dttm, discharge_dttm) / 1440.0 AS hosp_los_day
FROM base;
SELECT * FROM hospitalization
""")

# ECMO

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW ecmo AS
    SELECT
        REGEXP_REPLACE(hospitalization_id, '_[^_]+$', '') as hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        TRUE as ecmo
    FROM '{ecmo_path}'
    WHERE mcs_group = 'ecmo'
    GROUP BY hospitalizations_joined_id, date;
""")

ecmo_count = con.sql("SELECT COUNT(*) FROM ecmo").fetchone()[0]
if ecmo_count == 0:
    raise ValueError("ecmo view has 0 rows — check ecmo_path and mcs_group filter")
else:
    print(f"ecmo: {ecmo_count} rows")

con.sql("SELECT * FROM ecmo")

# Acidosis (pH < 7.2)

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW ph AS
    SELECT
        hospitalizations_joined_id,
        DATE(lab_collect_dttm) as date,
        MIN(lab_value_numeric) as ph_min_arterial_or_venous
    FROM '{labs_path}'
    WHERE lab_category IN ('ph_arterial', 'ph_venous')
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM ph
""")

# SPO2 daily min

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW spo2_min AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        MIN(vital_value) as spo2_min
    FROM '{vitals_path}'
    WHERE LOWER(vital_category) = 'spo2'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM spo2_min
""")

# PCO2

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW pco2 AS
    SELECT
        hospitalizations_joined_id,
        DATE(lab_collect_dttm) as date,
        MIN(lab_value_numeric) as pco2_min_arterial_or_venous
    FROM '{labs_path}'
    WHERE lower(lab_category) IN ('pco2_venous', 'pco2_arterial')
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM pco2
""")

# SUP

In [ ]:
con.sql(f"SELECT * FROM '{meds_continuous_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW sup AS
WITH continuous AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as sup
    FROM '{meds_continuous_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    )
),
intermittent AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as sup
    FROM '{meds_intermittent_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    )
),
combined AS (
    SELECT * FROM continuous
    UNION ALL
    SELECT * FROM intermittent
)
SELECT *
FROM combined
GROUP BY hospitalizations_joined_id, date, sup;
SELECT * FROM sup
""")

# Paralysis

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW paralytic_meds AS
WITH continuous AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytic_meds
    FROM '{meds_continuous_path}'
    WHERE med_group = 'paralytics'
        AND med_dose != '0'
),
intermittent AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytic_meds
    FROM '{meds_intermittent_path}'
    WHERE med_group = 'paralytics'
        AND med_dose != '0'
),
combined AS (
    SELECT * FROM continuous
    UNION ALL
    SELECT * FROM intermittent
)
SELECT *
FROM combined
GROUP BY hospitalizations_joined_id, date, paralytic_meds;
SELECT * FROM paralytic_meds
""")

# Vent data daily aggregate

In [ ]:
# Get vent info
con.sql(f"""
CREATE OR REPLACE TEMP VIEW vent_daily AS
    SELECT 
        hospitalizations_joined_id,
        DATE(recorded_date) AS date,
        COUNT(recorded_hour) AS daily_hours_on_vent,
        MIN(fio2_set) as vent_fio2_min,
        MAX(fio2_set) as vent_fio2_max,
        MIN(peep_set) AS vent_peep_min,
        MAX(peep_set) AS vent_peep_max
    FROM '{hourly_resp_path}'
    WHERE device_category = 'imv'
        AND vent_episode_id = '1'
        AND vent_episode_duration_hours >= '24'
        AND location_category = 'icu'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM vent_daily
""")

# Prone

In [ ]:
con.sql(f"SELECT * FROM '{position_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW prone AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        TRUE as prone_position
    FROM '{position_path}'
    WHERE position_category = 'prone'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM prone
""")

# Vent Mode

In [ ]:
con.sql(f"SELECT * FROM '{resp_supp_path}'")

In [ ]:
con.sql(f"SELECT distinct mode_category FROM '{resp_supp_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW vent_mode AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        MODE(mode_category) primary_mode_category
    FROM '{resp_supp_path}'
    WHERE device_category ILIKE '%imv%'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM vent_mode
""")

## Get observed tidal volumes

In [ ]:
# NOTE (Task 1): tidal_volume_obs is now fetched at the correct raw timepoint
# via raw_vt_df (see cell above). This view is retained for reference but is
# no longer used in the merge step below.
con.sql(f"""
CREATE OR REPLACE TEMP VIEW tidal_volume_obs AS
    SELECT
        hospitalizations_joined_id,
        recorded_dttm as tidal_obs_dttm,
        tidal_volume_obs
    FROM '{resp_supp_path}'
    WHERE tidal_volume_obs IS NOT NULL;
SELECT * FROM tidal_volume_obs
""")

# Covid

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW covid AS
    SELECT DISTINCT ON (mdm_link_id, date)
        patient_id as mdm_link_id,
        DATE(collect_dttm) as date,
        DATE(collect_dttm) as covid_date,
        TRUE as covid
    FROM '{micro_non_culture_path}'
    WHERE organism_category ILIKE '%sars%'
        AND LOWER(result_category) = 'detected';
SELECT * FROM covid
""")

# Hourly Paired PF Ratio

In [ ]:
po2_arterial = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    DATE(lab_collect_dttm) AS po2_date,
    HOUR(lab_collect_dttm) AS po2_hour,
    lab_collect_dttm AS hour_ts,
    lab_value_numeric AS po2_arterial,
    reference_unit
FROM '{labs_path}'
WHERE lab_category = 'po2_arterial'
""").df()
po2_arterial

In [ ]:
fio2_set_df = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    recorded_date AS date,
    recorded_hour AS hour,
    MAKE_TIMESTAMP(
        YEAR(recorded_date), MONTH(recorded_date), DAY(recorded_date),
        recorded_hour, 0, 0
    ) AS hour_ts,
    vent_episode_id,
    device_category,
    fio2_set
FROM '{hourly_resp_path}'
WHERE device_category = 'imv'
    AND vent_episode_id = '1'
    AND vent_episode_duration_hours >= '24'
    AND location_category = 'icu'
""").df()
fio2_set_df

In [ ]:
# merge_asof requires sorted data
fio2_set_df = fio2_set_df.sort_values('hour_ts')
po2_arterial = po2_arterial.sort_values('hour_ts')

po2_arterial['hour_ts'] = po2_arterial['hour_ts'].dt.tz_localize(None)

pf_ratio_df = pd.merge_asof(
    fio2_set_df,
    po2_arterial,
    by='hospitalizations_joined_id',
    on='hour_ts',
    direction='nearest',
    tolerance=pd.Timedelta('1 hour')
)

pf_ratio_df['pf_ratio'] = pf_ratio_df['po2_arterial'] / pf_ratio_df['fio2_set']
pf_ratio_df = pf_ratio_df[pf_ratio_df['pf_ratio'].notna()]

pf_ratio_df

In [ ]:
con.register('pf_ratio_df', pf_ratio_df)
con.sql(f"""
CREATE OR REPLACE TEMP VIEW pf_ratio_paired AS
    SELECT
        hospitalizations_joined_id,
        date(date) as date,
        MIN(pf_ratio) AS pf_ratio_paired_min
    FROM pf_ratio_df
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM pf_ratio_paired
""")

# LAPS2

## Save vent hospitalizations for LAPS2 export via R function

In [ ]:
con.sql(f"""
    COPY (
        SELECT
            hospitalizations_joined_id,
            ANY_VALUE(mdm_link_id) AS mdm_link_id,
            ANY_VALUE(mdm_link_id) AS patient_id
        FROM data
        GROUP BY hospitalizations_joined_id
    ) TO 'laps2_hospitalizations.parquet' (FORMAT PARQUET)
""")

## Call laps2 function and create laps2 view

In [ ]:
# !run_laps2.bat
# shutil.move('laps2_data.parquet', f'{output_path}/laps2_data.parquet')
# shutil.move('laps2_hospitalizations.parquet', f'{output_path}/laps2_hospitalizations.parquet')
# shutil.move('status.jsonl', f'{output_path}/status.jsonl')

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW laps2 AS
    SELECT DISTINCT ON (hospitalizations_joined_id, recorded_date)
        hospitalizations_joined_id,
        recorded_date as date,
        laps2
    FROM '{output_path}/laps2_data.parquet';
SELECT * FROM laps2
""")

# Merge duckdb tables

In [ ]:
print(con.sql(f"SELECT COUNT(*) FROM data"))
con.sql(f"""
CREATE OR REPLACE TABLE final_df AS
    SELECT
        data.*,
        patient.* EXCLUDE (mdm_link_id),
        height.height_cm,
        hospitalization.* EXCLUDE (hospitalizations_joined_id),
        vent_daily.* EXCLUDE (hospitalizations_joined_id, date),
        vent_mode.primary_mode_category,
        ph.ph_min_arterial_or_venous,
        pco2.pco2_min_arterial_or_venous,
        spo2_min.spo2_min,
        prone.prone_position,
        sup.sup,
        ecmo.ecmo,
        paralytic_meds.paralytic_meds,
        laps2.laps2,
        pf_ratio_paired.pf_ratio_paired_min
    FROM data
    LEFT JOIN patient USING (mdm_link_id)
    LEFT JOIN height USING (mdm_link_id)
    LEFT JOIN hospitalization USING (hospitalizations_joined_id)
    LEFT JOIN vent_daily USING (hospitalizations_joined_id, date)
    LEFT JOIN vent_mode USING (hospitalizations_joined_id, date)
    LEFT JOIN ph USING (hospitalizations_joined_id, date)
    LEFT JOIN pco2 USING (hospitalizations_joined_id, date)
    LEFT JOIN spo2_min USING (hospitalizations_joined_id, date)
    LEFT JOIN prone USING (hospitalizations_joined_id, date)
    LEFT JOIN sup USING (hospitalizations_joined_id, date)
    LEFT JOIN ecmo USING (hospitalizations_joined_id, date)
    LEFT JOIN paralytic_meds USING (hospitalizations_joined_id, date)
    LEFT JOIN laps2 USING (hospitalizations_joined_id, date)
    LEFT JOIN pf_ratio_paired USING (hospitalizations_joined_id, date);
SELECT * FROM final_df
""")
print(con.sql(f"SELECT COUNT(*) FROM final_df"))
# Make sure row numbers stay the same

In [ ]:
data = con.sql(f"SELECT * FROM final_df ORDER BY hospitalizations_joined_id, date").df()
data

# BMI

In [ ]:
weight = con.sql(f"""
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) AS date,
        vital_value AS weight_kg
    FROM '{vitals_path}'
    WHERE vital_category = 'weight_kg'
        AND vital_value IS NOT NULL
    ORDER BY recorded_dttm
""").df()

data = pd.merge_asof(
    data.sort_values('date'),
    weight.sort_values('date'),
    by='hospitalizations_joined_id',
    on='date',
    direction='nearest',
    allow_exact_matches=True
)

data = data.sort_values(by=['hospitalizations_joined_id', 'date'])
data['bmi'] = round(data['weight_kg'] / (data['height_cm'] / 100) ** 2, 2)
data

# Merge raw tidal volume into data (Task 1)

In [ ]:
# Replace waterfall tidal_volume_set with raw values from raw_vt_df.
# raw_vt_df has one row per (hospitalizations_joined_id, date) with
# tidal_volume_set and tidal_volume_obs at the clinically correct timepoint.
#
# NOTE: No pd.to_datetime() conversion — both data["date"] and raw_vt_df["date"]
# originate from the same DuckDB DATE column so they are already the same type.
# Explicit conversion would change datetime64[us] -> datetime64[ns] and break
# Cell 68's merge_asof (which uses df["date"] still in DuckDB native type).

data = data.drop(columns=['tidal_volume_set'])

data = data.merge(
    raw_vt_df[['hospitalizations_joined_id', 'date', 'tidal_volume_set', 'tidal_volume_obs']],
    on=['hospitalizations_joined_id', 'date'],
    how='left'
)

data = data.sort_values(by=['hospitalizations_joined_id', 'date']).reset_index(drop=True)

assert not data.duplicated(subset=['hospitalizations_joined_id', 'date']).any(), \
    "Duplicate (hospitalization, date) rows after raw Vt merge — check raw_vt_query"

print(f"Rows: {len(data)}")
print(f"tidal_volume_set non-null: {data['tidal_volume_set'].notna().sum()} "
      f"({data['tidal_volume_set'].notna().mean()*100:.1f}%)")
data[['hospitalizations_joined_id', 'date', 'tidal_volume_set', 'tidal_volume_obs']].head(5)


In [ ]:
# when is tidal_obs and not tidal_vol set > that is the non-volume control number

In [ ]:
data[['hospitalizations_joined_id', 'date', 'tidal_volume_obs', 'tidal_volume_set']]

## Merge in covid data within 2 weeks from first patient day. Carry forward?

In [ ]:
df = con.sql("""
             SELECT *
             FROM covid
             ORDER BY date
""").df().dropna()

intubation = data.sort_values(by='date').drop_duplicates(subset='hospitalizations_joined_id', keep='first')

# Merge within 14 days of intubation
intubation = pd.merge_asof(
    intubation,
    df,
    by='mdm_link_id',
    on='date',
    direction='backward',
    tolerance=pd.Timedelta('14D'),
    allow_exact_matches=True
)
intubation = intubation.sort_values(by=['hospitalizations_joined_id', 'date'])
print(f'Hospitalizations positive for covid: {len(intubation[intubation['covid'] == True])}')

intubation[intubation['covid'] == True][['hospitalizations_joined_id', 'mdm_link_id', 'date', 'covid_date', 'covid']]

In [ ]:
# Merge back onto main dataframe
data = data.merge(intubation[['hospitalizations_joined_id', 'covid']], on='hospitalizations_joined_id', how='left')
data[data['covid'] == True]

# Pandas engineering

In [ ]:
# Map hospital IDs
hosp_ids = {
    'Woodwinds' : 'Hospital 1',
    'St. Johns' : 'Hospital 2',
    'Southdale' : 'Hospital 3',
    'Riverside' : 'Hospital 4',
    'Ridges' : 'Hospital 5',
    'Ranges/Hibbing' : 'Hospital 6',
    'Northland' : 'Hospital 7',
    'Lakes' : 'Hospital 8',
    'Grand Itasca' : 'delete',
    'UMMC' : 'Hospital Ref',
    'St. Joes' : 'delete',
    'Bethesda' : 'delete'
}
data['hospital_id'] = data['hospital_id'].map(hosp_ids)
data[['hospital_id']]

In [ ]:
# Get mortality
data['mortality_inhospital'] = (data['death_date'].dt.date <= data['discharge_dttm'].dt.date).astype(bool)

In [ ]:
# Check for vent duration of at least 48hr for sup model
data['vent_episode_duration_48hr_min'] = (data['vent_episode_duration_hours'] >= 48).astype(bool)

In [ ]:
# Fillna meds / prone / sup
data['paralytic_meds'] = data['paralytic_meds'].fillna(False)
data['prone_position'] = data['prone_position'].fillna(False)
data['sup'] = data['sup'].fillna(False)

# COVID - fill in False
data['covid'] = data['covid'].fillna(False)

In [ ]:
# LTVV outcomes — derived from raw tidal_volume_set (Task 1).
# Rows where tidal_volume_set is NaN (no raw Vt found at correct timepoint)
# will compute as 0 here; Cell 82 drops those rows before the final parquet.

# Primary outcomes: raw set Vt only
data['tidal_volume_set_ibw'] = data['tidal_volume_set'] / data['ibw']
data['ltvv_8'] = (data['tidal_volume_set_ibw'] <= 8).astype(int)
data['ltvv_6'] = (data['tidal_volume_set_ibw'] <= 6).astype(int)

# Secondary fallback outcomes: raw set-or-obs
data['tidal_volume_set_or_obs'] = data['tidal_volume_set'].fillna(data['tidal_volume_obs'])
data['tidal_volume_set_or_obs_ibw'] = data['tidal_volume_set_or_obs'] / data['ibw']
data['ltvv_tidal_volume_set_or_obs_8'] = (data['tidal_volume_set_or_obs_ibw'] <= 8).astype(int)
data['ltvv_tidal_volume_set_or_obs_6'] = (data['tidal_volume_set_or_obs_ibw'] <= 6).astype(int)

print(f"ltvv_6 rate (pre-exclusion): {data['ltvv_6'].mean()*100:.1f}%")
print(f"ltvv_8 rate (pre-exclusion): {data['ltvv_8'].mean()*100:.1f}%")
data.head(5)

## Carry forward sf_ratio min, pf ratio min, ph min, and pco2 max up to 3 days

In [ ]:
# Carry forward ph and pco2 up to 3 days max
data = data.sort_values(by=['hospitalizations_joined_id', 'date'])

data['ph_min_arterial_or_venous'] = (
    data.groupby('hospitalizations_joined_id')['ph_min_arterial_or_venous']
    .ffill(limit=3)
)
data['pco2_min_arterial_or_venous'] = (
    data.groupby('hospitalizations_joined_id')['pco2_min_arterial_or_venous']
    .ffill(limit=3)
)

# Carry forward pf_ratio_paired_min up to 3 days max
data['pf_ratio_paired_min'] = (
    data.groupby('hospitalizations_joined_id')['pf_ratio_paired_min']
    .ffill(limit=3)
)

data

# Exclusion/Inclusion

In [ ]:
# Remove tracheostomy rows
print(f'Rows before dropping trach: {len(data)}')
data = data[data['tracheostomy'] != 1].copy()
data['tracheostomy'] = data['tracheostomy'].fillna(False).astype(bool)
print(f'Rows after dropping trach: {len(data)}')

data

In [ ]:
# Remove ECMO days
print(len(data))
data['ecmo'] = data['ecmo'].fillna(False)
data = data[data['ecmo'] == False].copy()
print(len(data))

In [ ]:
## Leftover exclusions
# Make sure 18+
print(f'Starting row n: {len(data)}')
data = data[data['age'] >= 18]
print(f'Age: {len(data)}')

# Remove st. joes, bethesda, and grand itasca
data = data[data['hospital_id'] != 'delete']
print(f'Hospital ID: {len(data)}')

# Remove weird stuff
data = data.dropna(subset=['prov_npi_shifted'])
data = data[~data['prov_npi_shifted'].isin(['None', 'none', 'nan'])]
print(f'Provider: {len(data)}')

data

In [ ]:
# Keep only if tidal_volume_set not null
print(f"Before dropping na: {len(data)}")
data = data[~data['tidal_volume_set'].isna()]
print(f"After dropping na: {len(data)}")

data

# Missingness

In [ ]:
print(len(data))
data.isna().sum()

# ARDS

In [ ]:
# !run_ards.bat
# shutil.move('ards_classifier_cohort.csv', f'{output_path}/ards_classifier_cohort.csv')
# shutil.move('status.jsonl', f'{output_path}/status.jsonl')
# shutil.move('all_hosp_ids_on_vent.parquet', f'{output_path}/all_hosp_ids_on_vent.parquet')

In [ ]:
ards_classifier_cohort = pd.read_csv(f'{output_path}/ards_classifier_cohort.csv', index_col=0)
ards_classifier_cohort["hospitalizations_joined_id"] = ards_classifier_cohort["hospitalization_id"].str.replace(
    r"^([^_]+_[^_]+).*", r"\1", regex=True)
print(len(ards_classifier_cohort))

# some duplicate rows remain - drop
ards_classifier_cohort = ards_classifier_cohort.drop_duplicates()
print(len(ards_classifier_cohort))

# Some duplicate hospitalizations_joined_id > only difference is cardiac_arrest_primary_dx column
# Remove duplicate cardiac_arres_primary_dx
ards_classifier_cohort = ards_classifier_cohort.sort_values('cardiac_arrest_primary_dx', ascending = False)\
    .drop_duplicates('hospitalizations_joined_id')
print(len(ards_classifier_cohort))

# Merge onto data
print(len(data))
data = data.merge(ards_classifier_cohort[['hospitalizations_joined_id', 'cohort_eligible']], on='hospitalizations_joined_id', how='inner')
data['ards_eligible'] = data['cohort_eligible']
data

# Save final dataframe

## Table 1

### Definition to create table 1

In [ ]:
def create_table1(df, continuous_vars, categorical_vars, var_order=None):
    """
    Create a Table 1 summary statistics table

    Parameters
    ----------
    df : pandas DataFrame
        The patient data
    continuous_vars : dict
        {column_name: display_label} for continuous vars
        e.g., {'age': 'Age, median [Q1,Q3]'}
    categorical_vars : dict
        {column_name: display_label} for categorical vars
        e.g., {'race': 'Race, n (%)'}
    var_order : list | None
        Exact row order; use 'n' for the total count.
        Example: ['n', 'age', 'bmi_calc', 'race_category', 'sex_category', ...]

    Returns
    -------
    pandas DataFrame with columns ['Variable','Category','Value']
    """

    results = []

    # Total N
    total_n = len(df)

    def _add_n():
        results.append({'Variable': 'n', 'Category': '', 'Value': str(total_n)})

    def _add_cont(var):
        if var in df.columns:
            label = continuous_vars[var]
            median = df[var].median()
            q1 = df[var].quantile(0.25)
            q3 = df[var].quantile(0.75)
            value = f"{median:.1f} [{q1:.1f},{q3:.1f}]"
            results.append({'Variable': label, 'Category': '', 'Value': value})

    def _add_cat(var):
        if var in df.columns:
            label = categorical_vars[var]
            # header row
            results.append({'Variable': label, 'Category': '', 'Value': ''})
            # category rows
            value_counts = df[var].value_counts()
            for category, count in value_counts.items():
                pct = (count / total_n) * 100
                value = f"{count} ({pct:.1f})"
                results.append({'Variable': '', 'Category': str(category), 'Value': value})

    if var_order is not None:
        # Follow the exact order provided
        for key in var_order:
            if key == 'n':
                _add_n()
            elif key in continuous_vars:
                _add_cont(key)
            elif key in categorical_vars:
                _add_cat(key)
            # else: silently ignore unknown keys
    else:
        # Original behavior (keep as-is)
        _add_n()
        for var, _label in continuous_vars.items():
            _add_cont(var)
        for var, _label in categorical_vars.items():
            _add_cat(var)

    return pd.DataFrame(results)

### Create table 1

In [ ]:
table1_data = data.groupby(['hospitalizations_joined_id']).agg(
    age=('age', 'first'),
    bmi=('bmi', 'median'),
    elix_vw=('elix_vw', 'median'),
    pf_ratio_paired_min=('pf_ratio_paired_min', 'median'),
    pco2_min_arterial_or_venous=('pco2_min_arterial_or_venous', 'median'),
    ph_min_arterial_or_venous=('ph_min_arterial_or_venous', 'median'),
    laps2=('laps2', 'median'),
    vent_episode_duration_hours=('vent_episode_duration_hours', 'first'),
    hosp_los_day=('hosp_los_day', 'first'),
    race_category=('race_category', 'first'),
    sex_category=('sex_category', 'first'),
    admit_year=('admit_year', 'first'),
    hospital_id=('hospital_id', 'first'),
    mortality_inhospital=('mortality_inhospital', 'first'),
    ards_eligible=('ards_eligible', 'first')
).reset_index()

table1_data

In [ ]:
# Define variables for Table 1
continuous_vars = {
    'age': 'Age, median [Q1,Q3]',
    'bmi': 'BMI, median [Q1,Q3]',
    'elix_vw': 'Elixhauser, median [Q1,Q3]',
    'pf_ratio_paired_min': 'PF Ratio (paired), median [Q1,Q3]',
    'pco2_min_arterial_or_venous': 'pCO2 (arterial or venous), median [Q1,Q3]',
    'ph_min_arterial_or_venous': 'pH (arterial or venous), median [Q1,Q3]',
    'laps2': 'Laps2, median [Q1,Q3]',
    'vent_episode_duration_hours': 'Vent episode duration (hr), median [Q1,Q3]',
    'hosp_los_day': 'Hospitalization length (days), median [Q1,Q3]'
}

categorical_vars = {
    'race_category': 'Race, n (%)',
    'sex_category': 'Sex, n (%)',
    'admit_year': 'Year, n (%)',
    'hospital_id': 'Hospital, n (%)',
    'mortality_inhospital': 'In-hospitalization mortality, n (%)'
}

var_order = ['n',
             'age', 'race_category', 'sex_category',
             'bmi', 'elix_vw', 'laps2',
             'pf_ratio_paired_min', 'pco2_min_arterial_or_venous', 'ph_min_arterial_or_venous', 'vent_episode_duration_hours',
             'hosp_los_day', 'mortality_inhospital', 'hospital_id', 'admit_year'
]

# Generate Table 1 (overall only)
table1 = create_table1(table1_data, continuous_vars, categorical_vars, var_order)
table1.to_excel(f'{output_path}/table1.xlsx', index=False)

# Generate Table 1 stratified by ARDS cohort eligibility
table1_stratified = table1.rename(columns={'Value': 'Overall'}).copy()

for cohort_value in sorted(table1_data['ards_eligible'].dropna().unique()):
    cohort_df = table1_data[table1_data['ards_eligible'] == cohort_value]
    if cohort_df.empty:
        continue

    if cohort_value in (1, True):
        cohort_label = 'ARDS cohort'
    elif cohort_value in (0, False):
        cohort_label = 'Non-ARDS cohort'
    else:
        cohort_label = f'ards_eligible={cohort_value}'

    cohort_table = create_table1(cohort_df, continuous_vars, categorical_vars, var_order)
    cohort_table = cohort_table.rename(columns={'Value': cohort_label})
    table1_stratified = table1_stratified.merge(
        cohort_table[['Variable', 'Category', cohort_label]],
        on=['Variable', 'Category'],
        how='left'
    )

table1_stratified.to_excel(f'{output_path}/table1_stratified_by_ards_eligible.xlsx', index=False)

table1, table1_stratified

# Provider counts

In [ ]:
provider_count_df = data.sort_values(by=['hospitalizations_joined_id', 'date'])
provider_count_df = provider_count_df.drop_duplicates(subset='hospitalizations_joined_id')

counts = provider_count_df['prov_npi_shifted'].value_counts()

median_count = counts.median()
q1 = counts.quantile(0.25)
q3 = counts.quantile(0.75)
iqr = q3 - q1

print(f"Median count: {median_count}")
print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")

# Save final data to parquet

In [ ]:
data.to_parquet(f'{output_path}/LPV_final_data.parquet')
print(f'Hospitalization Number: {len(data['hospitalizations_joined_id'].unique())}')
print(f'Patient Number: {len(data['mdm_link_id'].unique())}')
print(f'Provider Number: {len(data['prov_npi_shifted'].unique())}')
print(f'Date Number: {len(data)}')

In [ ]:
ards_data = data[data['ards_eligible'] == 1]
print(f'Hospitalization Number: {len(ards_data['hospitalizations_joined_id'].unique())}')
print(f'Patient Number: {len(ards_data['mdm_link_id'].unique())}')
print(f'Provider Number: {len(ards_data['prov_npi_shifted'].unique())}')
print(f'Date Number: {len(ards_data)}')
ards_data